# Phase 1: Data Protocol And Patch Dataset

Goal: build a reproducible 16-band CAVE patch dataset with a fixed scene split.

Deliverables:
- `dataset_patches.npz`
- `split_metadata.json`
- one qualitative sanity-check figure


# Learnable MSFA Research Track

This notebook is part of the 10-day publication-oriented extension track.

## Run discipline
- Keep one experiment purpose per notebook.
- Save metrics and checkpoints to the configured project root.
- Do not silently change data splits or loss definitions across notebooks.
- When a result becomes final, export both numeric artifacts and a figure/table asset.


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted.")
except Exception as exc:
    print("Drive mount skipped:", exc)


In [ ]:
import json
import random
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path("/content/drive/MyDrive/Msa-Osp")
ZIP_PATH = PROJECT_ROOT / "complete_ms_data.zip"
UNZIP_DIR = Path("/content/unzipped_data")
PATCH_PATH = PROJECT_ROOT / "dataset_patches.npz"
SPLIT_PATH = PROJECT_ROOT / "split_metadata.json"

PATCH_SIZE = 128
PATCHES_PER_SCENE = 40
BAND_COUNT = 16
TRAIN_SCENES = 24

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)


In [ ]:
if not UNZIP_DIR.exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(UNZIP_DIR)

scene_names = sorted(d.name for d in UNZIP_DIR.iterdir() if d.is_dir())
print("Total scenes:", len(scene_names))
print("First five scenes:", scene_names[:5])


In [ ]:
def load_scene(scene_root, band_count=BAND_COUNT):
    subdirs = [d for d in scene_root.iterdir() if d.is_dir()]
    if not subdirs:
        raise ValueError(f"No nested folder found inside {scene_root}")

    scene_dir = subdirs[0]
    band_files = sorted(
        [p for p in scene_dir.iterdir() if p.suffix.lower() == ".png" and "_ms_" in p.name],
        key=lambda p: int(p.stem.split("_ms_")[1]),
    )[:band_count]

    bands = []
    for path in band_files:
        image = np.array(Image.open(path), dtype=np.float32)
        if image.ndim == 3:
            image = image[:, :, 0]
        bands.append(image)

    cube = np.stack(bands, axis=-1)
    cube /= 65535.0 if cube.max() > 255 else 255.0
    return cube.astype(np.float32)


def extract_random_patches(cube, patch_size=PATCH_SIZE, patches_per_scene=PATCHES_PER_SCENE):
    h, w, _ = cube.shape
    patches = []
    for _ in range(patches_per_scene):
        top = random.randint(0, h - patch_size)
        left = random.randint(0, w - patch_size)
        patches.append(cube[top:top + patch_size, left:left + patch_size, :])
    return patches


def to_rgb(cube):
    rgb = cube[:, :, [5, 10, 15]]
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
    return rgb


In [ ]:
train_scenes = scene_names[:TRAIN_SCENES]
val_scenes = scene_names[TRAIN_SCENES:]

train_patches = []
val_patches = []

for scene_name in train_scenes:
    cube = load_scene(UNZIP_DIR / scene_name)
    train_patches.extend(extract_random_patches(cube))

for scene_name in val_scenes:
    cube = load_scene(UNZIP_DIR / scene_name)
    val_patches.extend(extract_random_patches(cube))

train_data = np.asarray(train_patches, dtype=np.float32)
val_data = np.asarray(val_patches, dtype=np.float32)

PATCH_PATH.parent.mkdir(parents=True, exist_ok=True)
SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(PATCH_PATH, train=train_data, val=val_data)
SPLIT_PATH.write_text(
    json.dumps(
        {
            "seed": SEED,
            "band_count": BAND_COUNT,
            "patch_size": PATCH_SIZE,
            "patches_per_scene": PATCHES_PER_SCENE,
            "train_scenes": train_scenes,
            "val_scenes": val_scenes,
        },
        indent=2,
    )
)

print("Train shape:", train_data.shape)
print("Val shape:", val_data.shape)


In [ ]:
example = train_data[0]
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(to_rgb(example))
plt.title("Example fake RGB")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.plot(example[example.shape[0] // 2, example.shape[1] // 2, :])
plt.title("Center-pixel spectrum")
plt.xlabel("Band")
plt.ylabel("Normalized intensity")
plt.tight_layout()
plt.show()
